In [1]:
import pandas as pd
import os
import json

# This prints the path where pulse data is
base_path = r"C:\Users\himan\OneDrive\Desktop\phonepe-pulse-analytics\pulse\data"
print("Pulse data path:", base_path)
print("Folders inside:", os.listdir(base_path))


Pulse data path: C:\Users\himan\OneDrive\Desktop\phonepe-pulse-analytics\pulse\data
Folders inside: ['aggregated', 'map', 'top']


In [2]:
# Let's look inside aggregated folder
agg_path = os.path.join(base_path, 'aggregated')
print("Inside aggregated:", os.listdir(agg_path))

# Look inside transaction folder
txn_path = os.path.join(agg_path, 'transaction', 'country', 'india', 'state')
print("\nStates available:", os.listdir(txn_path))

Inside aggregated: ['insurance', 'transaction', 'user']

States available: ['andaman-&-nicobar-islands', 'andhra-pradesh', 'arunachal-pradesh', 'assam', 'bihar', 'chandigarh', 'chhattisgarh', 'dadra-&-nagar-haveli-&-daman-&-diu', 'delhi', 'goa', 'gujarat', 'haryana', 'himachal-pradesh', 'jammu-&-kashmir', 'jharkhand', 'karnataka', 'kerala', 'ladakh', 'lakshadweep', 'madhya-pradesh', 'maharashtra', 'manipur', 'meghalaya', 'mizoram', 'nagaland', 'odisha', 'puducherry', 'punjab', 'rajasthan', 'sikkim', 'tamil-nadu', 'telangana', 'tripura', 'uttar-pradesh', 'uttarakhand', 'west-bengal']


In [3]:
# Read one JSON file to see what's inside
sample_file = os.path.join(txn_path, 'andhra-pradesh', '2022', '1.json')

with open(sample_file, 'r') as f:
    data = json.load(f)

print("Keys in JSON:", data.keys())
print("\nFull data:")
print(json.dumps(data, indent=2))

Keys in JSON: dict_keys(['success', 'code', 'data', 'responseTimestamp'])

Full data:
{
  "success": true,
  "code": "SUCCESS",
  "data": {
    "from": 1640975400000,
    "to": 1648405800000,
    "transactionData": [
      {
        "name": "Peer-to-peer payments",
        "paymentInstruments": [
          {
            "type": "TOTAL",
            "count": 288488560,
            "amount": 1089629182596.4781
          }
        ]
      },
      {
        "name": "Merchant payments",
        "paymentInstruments": [
          {
            "type": "TOTAL",
            "count": 242031836,
            "amount": 169419685917.19562
          }
        ]
      },
      {
        "name": "Recharge & bill payments",
        "paymentInstruments": [
          {
            "type": "TOTAL",
            "count": 51224170,
            "amount": 34473036841.83001
          }
        ]
      },
      {
        "name": "Financial Services",
        "paymentInstruments": [
          {
            "type"

In [4]:
# Read ALL aggregated transaction data from all states and years
all_txn_data = []

for state in os.listdir(txn_path):
    state_path = os.path.join(txn_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for quarter_file in os.listdir(year_path):
            file_path = os.path.join(year_path, quarter_file)
            quarter = quarter_file.replace('.json', '')
            
            with open(file_path, 'r') as f:
                data = json.load(f)
            
            if data['data'] and data['data']['transactionData']:
                for txn in data['data']['transactionData']:
                    all_txn_data.append({
                        'state': state,
                        'year': int(year),
                        'quarter': int(quarter),
                        'transaction_type': txn['name'],
                        'transaction_count': txn['paymentInstruments'][0]['count'],
                        'transaction_amount': txn['paymentInstruments'][0]['amount']
                    })

df_agg_txn = pd.DataFrame(all_txn_data)
print("Shape:", df_agg_txn.shape)
print("\nFirst 5 rows:")
print(df_agg_txn.head())

Shape: (5034, 6)

First 5 rows:
                       state  year  quarter          transaction_type  \
0  andaman-&-nicobar-islands  2018        1  Recharge & bill payments   
1  andaman-&-nicobar-islands  2018        1     Peer-to-peer payments   
2  andaman-&-nicobar-islands  2018        1         Merchant payments   
3  andaman-&-nicobar-islands  2018        1        Financial Services   
4  andaman-&-nicobar-islands  2018        1                    Others   

   transaction_count  transaction_amount  
0               4200        1.845307e+06  
1               1871        1.213866e+07  
2                298        4.525072e+05  
3                 33        1.060142e+04  
4                256        1.846899e+05  


In [5]:
# Save to CSV first
df_agg_txn.to_csv(r'C:\Users\himan\OneDrive\Desktop\phonepe-pulse-analytics\data\processed\agg_transactions.csv', index=False)
print("Saved agg_transactions.csv")
print("Shape:", df_agg_txn.shape)
print("\nUnique states:", df_agg_txn['state'].nunique())
print("Years covered:", sorted(df_agg_txn['year'].unique()))
print("Transaction types:", df_agg_txn['transaction_type'].unique())

Saved agg_transactions.csv
Shape: (5034, 6)

Unique states: 36
Years covered: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Transaction types: <StringArray>
['Recharge & bill payments',    'Peer-to-peer payments',
        'Merchant payments',       'Financial Services',
                   'Others']
Length: 5, dtype: str


In [6]:
# Read ALL aggregated user data
user_path = os.path.join(agg_path, 'user', 'country', 'india', 'state')
all_user_data = []

for state in os.listdir(user_path):
    state_path = os.path.join(user_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for quarter_file in os.listdir(year_path):
            file_path = os.path.join(year_path, quarter_file)
            quarter = quarter_file.replace('.json', '')
            
            with open(file_path, 'r') as f:
                data = json.load(f)
            
            if data['data'] and data['data'].get('usersByDevice'):
                for device in data['data']['usersByDevice']:
                    all_user_data.append({
                        'state': state,
                        'year': int(year),
                        'quarter': int(quarter),
                        'device_brand': device['brand'],
                        'user_count': device['count'],
                        'percentage': device['percentage']
                    })

df_agg_user = pd.DataFrame(all_user_data)
df_agg_user.to_csv(r'C:\Users\himan\OneDrive\Desktop\phonepe-pulse-analytics\data\processed\agg_users.csv', index=False)
print("Saved agg_users.csv")
print("Shape:", df_agg_user.shape)
print("\nTop device brands:")
print(df_agg_user.groupby('device_brand')['user_count'].sum().sort_values(ascending=False).head(10))

Saved agg_users.csv
Shape: (6732, 6)

Top device brands:
device_brand
Xiaomi      869562617
Samsung     671603711
Vivo        625415019
Oppo        420250245
Others      282950234
Realme      219973222
Apple        95947314
Motorola     73340734
OnePlus      63677211
Huawei       57129693
Name: user_count, dtype: int64


In [7]:
# Read MAP transaction data (district level)
map_txn_path = os.path.join(base_path, 'map', 'transaction', 
                             'hover', 'country', 'india', 'state')
all_map_txn = []

for state in os.listdir(map_txn_path):
    state_path = os.path.join(map_txn_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for quarter_file in os.listdir(year_path):
            file_path = os.path.join(year_path, quarter_file)
            quarter = quarter_file.replace('.json', '')
            
            with open(file_path, 'r') as f:
                data = json.load(f)
            
            if data['data'] and data['data'].get('hoverDataList'):
                for district in data['data']['hoverDataList']:
                    all_map_txn.append({
                        'state': state,
                        'district': district['name'],
                        'year': int(year),
                        'quarter': int(quarter),
                        'transaction_count': district['metric'][0]['count'],
                        'transaction_amount': district['metric'][0]['amount']
                    })

df_map_txn = pd.DataFrame(all_map_txn)
df_map_txn.to_csv(r'C:\Users\himan\OneDrive\Desktop\phonepe-pulse-analytics\data\processed\map_transactions.csv', index=False)
print("Saved map_transactions.csv")
print("Shape:", df_map_txn.shape)
print("\nSample:")
print(df_map_txn.head())

Saved map_transactions.csv
Shape: (20604, 6)

Sample:
                       state                           district  year  \
0  andaman-&-nicobar-islands  north and middle andaman district  2018   
1  andaman-&-nicobar-islands             south andaman district  2018   
2  andaman-&-nicobar-islands                  nicobars district  2018   
3  andaman-&-nicobar-islands  north and middle andaman district  2018   
4  andaman-&-nicobar-islands             south andaman district  2018   

   quarter  transaction_count  transaction_amount  
0        1                442        9.316631e+05  
1        1               5688        1.256025e+07  
2        1                528        1.139849e+06  
3        2                825        1.317863e+06  
4        2               9395        2.394824e+07  


In [8]:
# Read MAP user data (district level)
map_user_path = os.path.join(base_path, 'map', 'user', 
                              'hover', 'country', 'india', 'state')
all_map_user = []

for state in os.listdir(map_user_path):
    state_path = os.path.join(map_user_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for quarter_file in os.listdir(year_path):
            file_path = os.path.join(year_path, quarter_file)
            quarter = quarter_file.replace('.json', '')
            
            with open(file_path, 'r') as f:
                data = json.load(f)
            
            if data['data'] and data['data'].get('hoverData'):
                for district, values in data['data']['hoverData'].items():
                    all_map_user.append({
                        'state': state,
                        'district': district,
                        'year': int(year),
                        'quarter': int(quarter),
                        'registered_users': values['registeredUsers'],
                        'app_opens': values['appOpens']
                    })

df_map_user = pd.DataFrame(all_map_user)
df_map_user.to_csv(r'C:\Users\himan\OneDrive\Desktop\phonepe-pulse-analytics\data\processed\map_users.csv', index=False)
print("Saved map_users.csv")
print("Shape:", df_map_user.shape)
print("\nSample:")
print(df_map_user.head())

Saved map_users.csv
Shape: (20608, 6)

Sample:
                       state                           district  year  \
0  andaman-&-nicobar-islands  north and middle andaman district  2018   
1  andaman-&-nicobar-islands             south andaman district  2018   
2  andaman-&-nicobar-islands                  nicobars district  2018   
3  andaman-&-nicobar-islands  north and middle andaman district  2018   
4  andaman-&-nicobar-islands             south andaman district  2018   

   quarter  registered_users  app_opens  
0        1               632          0  
1        1              5846          0  
2        1               262          0  
3        2               911          0  
4        2              8143          0  
